# San José 311 · Live weekly forecasting demo
Run all cells in Jupyter or Colab. The notebook includes aggregate Monday–Sunday counts for the ten districts, computed from the project’s cleaned records and archived district map. No request-level download or repository setup is required. The demonstration reproduces the selected Poisson specification and two baselines.

In [1]:
from io import StringIO
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import PoissonRegressor

### Weekly inputs
One row per Monday, one column per district. Expand the data cell to inspect the weekly counts.

In [2]:
weekly_csv = """week_start,1,2,3,4,5,6,7,8,9,10
2018-01-01,9,43,83,38,36,37,36,38,13,3
2018-01-08,23,48,85,53,31,45,31,28,21,15
2018-01-15,27,34,84,53,64,52,47,20,20,11
2018-01-22,41,48,89,45,40,45,41,33,21,15
2018-01-29,28,45,98,63,50,60,65,43,8,15
2018-02-05,21,50,81,57,31,50,60,27,24,9
2018-02-12,34,49,95,33,32,46,82,30,24,26
2018-02-19,25,34,74,33,36,32,115,22,17,7
2018-02-26,29,22,77,44,36,31,98,24,15,22
2018-03-05,31,27,85,61,46,38,77,34,19,20
2018-03-12,44,26,70,45,74,41,64,30,7,16
2018-03-19,27,36,57,36,57,41,47,21,8,6
2018-03-26,22,39,65,35,40,45,67,21,7,10
2018-04-02,70,37,89,29,39,46,92,31,11,16
2018-04-09,31,35,69,42,54,50,53,29,13,23
2018-04-16,32,50,67,54,48,36,98,25,23,22
2018-04-23,22,50,60,29,56,38,63,28,13,16
2018-04-30,41,34,76,23,58,42,71,21,21,15
2018-05-07,20,49,85,37,43,30,90,26,28,21
2018-05-14,27,38,70,27,30,30,101,26,23,8
2018-05-21,42,28,67,26,35,35,74,23,23,15
2018-05-28,25,52,112,35,38,31,81,30,17,14
2018-06-04,48,41,143,29,50,36,63,27,21,14
2018-06-11,25,46,123,40,55,39,62,28,16,20
2018-06-18,37,52,103,30,43,42,84,28,11,21
2018-06-25,32,35,104,44,74,47,93,42,17,21
2018-07-02,31,55,97,36,78,51,90,51,24,23
2018-07-09,30,62,114,42,60,61,69,39,29,15
2018-07-16,43,59,109,52,50,70,116,40,29,20
2018-07-23,42,51,103,44,54,60,75,34,25,26
2018-07-30,37,71,100,48,76,61,66,28,24,30
2018-08-06,34,62,91,43,58,63,71,41,29,17
2018-08-13,37,55,98,30,54,47,91,50,19,23
2018-08-20,35,52,85,32,63,57,75,43,20,20
2018-08-27,27,49,104,51,55,46,73,34,21,19
2018-09-03,39,80,88,51,45,35,78,49,30,16
2018-09-10,38,62,93,49,50,56,46,32,15,19
2018-09-17,33,45,68,44,61,45,68,35,34,12
2018-09-24,30,31,88,33,60,58,63,16,13,18
2018-10-01,34,48,122,34,40,52,42,51,34,16
2018-10-08,22,31,115,21,55,43,61,46,25,17
2018-10-15,12,47,97,30,43,36,50,36,14,25
2018-10-22,39,52,102,26,66,21,74,51,26,12
2018-10-29,35,42,107,28,46,25,62,49,15,15
2018-11-05,30,34,98,26,47,42,77,37,19,14
2018-11-12,15,37,77,29,36,34,50,39,17,17
2018-11-19,14,37,70,15,28,25,36,14,10,11
2018-11-26,31,65,130,52,38,35,89,40,12,22
2018-12-03,35,43,106,36,49,43,64,37,31,17
2018-12-10,16,51,126,27,62,33,58,33,11,20
2018-12-17,39,45,74,50,42,30,74,41,10,15
2018-12-24,30,33,46,34,21,25,44,10,7,9
2018-12-31,27,57,81,24,36,40,65,25,17,13
2019-01-07,32,47,89,34,48,47,98,27,23,19
2019-01-14,19,48,100,21,43,45,56,45,12,28
2019-01-21,36,58,76,24,65,24,66,37,9,20
2019-01-28,36,40,111,35,69,45,52,37,21,18
2019-02-04,28,42,84,26,52,28,58,33,11,9
2019-02-11,33,21,82,32,31,27,54,18,15,9
2019-02-18,41,59,57,31,28,31,73,35,11,16
2019-02-25,40,45,109,28,50,37,83,27,12,11
2019-03-04,14,24,64,8,20,13,28,16,9,12
2019-03-11,36,64,85,28,72,39,72,33,15,14
2019-03-18,41,42,81,32,49,34,74,35,20,30
2019-03-25,39,42,98,37,64,36,84,29,8,16
2019-04-01,33,61,91,37,67,58,82,35,20,17
2019-04-08,26,62,92,24,62,56,74,44,15,14
2019-04-15,46,33,113,26,55,43,87,46,16,18
2019-04-22,30,47,95,21,75,37,68,36,23,17
2019-04-29,53,51,116,22,72,50,75,32,20,23
2019-05-06,42,63,123,34,79,43,88,61,17,18
2019-05-13,31,35,76,49,68,38,43,48,22,20
2019-05-20,22,49,132,27,61,57,59,58,20,17
2019-05-27,20,48,118,29,65,67,53,44,26,21
2019-06-03,31,36,152,59,74,42,61,51,20,23
2019-06-10,27,40,127,26,78,27,61,32,34,24
2019-06-17,28,57,127,37,58,45,66,27,20,24
2019-06-24,32,47,140,39,74,45,52,41,22,16
2019-07-01,34,37,140,38,53,23,51,27,15,23
2019-07-08,52,43,142,20,55,67,71,31,10,14
2019-07-15,31,47,122,26,50,78,69,27,16,25
2019-07-22,46,42,124,36,49,52,57,16,13,19
2019-07-29,50,59,130,31,63,44,77,31,19,18
2019-08-05,46,40,115,38,55,76,68,39,15,15
2019-08-12,40,38,105,31,51,68,76,36,23,12
2019-08-19,49,55,100,22,52,51,79,19,17,13
2019-08-26,50,54,78,17,85,52,68,29,27,25
2019-09-02,43,53,115,35,51,68,88,19,54,23
2019-09-09,54,45,80,30,57,39,94,23,79,21
2019-09-16,29,51,86,24,62,39,79,25,96,21
2019-09-23,52,70,108,30,38,59,69,34,75,19
2019-09-30,21,44,110,27,51,41,62,27,73,26
2019-10-07,32,45,84,22,66,51,63,43,93,21
2019-10-14,23,41,84,22,56,34,61,29,72,8
2019-10-21,24,45,68,27,59,34,71,17,137,14
2019-10-28,32,44,84,25,44,22,46,27,86,11
2019-11-04,30,32,87,49,53,39,68,25,113,12
2019-11-11,29,34,101,32,59,29,46,27,118,12
2019-11-18,39,65,99,30,57,30,59,22,139,9
2019-11-25,13,21,93,18,26,27,23,13,136,7
2019-12-02,42,45,97,34,61,41,52,20,124,12
2019-12-09,11,43,117,35,44,38,64,24,157,15
2019-12-16,39,49,76,26,33,40,65,27,143,13
2019-12-23,14,31,57,20,34,12,45,33,81,12
2019-12-30,35,41,89,34,40,34,39,31,95,13
2020-01-06,28,61,97,26,68,67,60,21,291,9
2020-01-13,28,51,100,29,63,34,72,30,135,16
2020-01-20,45,62,117,31,53,40,58,31,162,25
2020-01-27,31,53,130,44,43,56,65,28,148,26
2020-02-03,43,41,102,32,45,45,73,40,155,16
2020-02-10,44,35,88,33,41,46,71,36,134,8
2020-02-17,67,40,102,18,44,42,42,32,182,12
2020-02-24,51,43,92,44,59,40,52,32,160,23
2020-03-02,70,48,106,16,46,34,71,33,141,12
2020-03-09,37,51,95,34,58,30,52,19,190,21
2020-03-16,13,16,67,14,30,20,29,11,134,12
2020-03-23,10,22,46,26,34,13,34,22,132,7
2020-03-30,22,23,86,15,30,21,32,23,146,9
2020-04-06,11,42,74,26,47,29,77,20,353,15
2020-04-13,21,26,104,30,57,38,43,42,269,12
2020-04-20,21,18,98,22,45,26,68,25,232,10
2020-04-27,38,75,105,33,62,37,59,23,237,18
2020-05-04,15,32,70,35,61,38,50,26,228,11
2020-05-11,24,26,117,26,75,17,85,24,235,9
2020-05-18,24,32,115,43,54,21,58,27,422,7
2020-05-25,24,24,120,21,56,35,53,20,281,8
2020-06-01,40,25,133,33,38,34,65,19,545,18
2020-06-08,29,35,111,42,41,24,55,25,299,27
2020-06-15,29,38,111,35,51,38,76,32,58,15
2020-06-22,38,39,94,33,67,38,50,34,10,13
2020-06-29,22,30,111,35,49,28,73,37,22,10
2020-07-06,29,43,135,35,46,31,58,25,30,10
2020-07-13,33,54,120,40,51,44,83,38,13,13
2020-07-20,29,60,120,32,81,67,81,50,29,10
2020-07-27,51,42,142,61,69,45,87,38,35,33
2020-08-03,42,63,108,33,61,56,75,39,19,14
2020-08-10,30,34,105,32,56,49,41,32,19,15
2020-08-17,20,41,117,33,45,30,59,30,44,18
2020-08-24,36,49,92,31,56,43,82,28,19,13
2020-08-31,48,43,91,32,54,29,72,40,12,9
2020-09-07,16,50,85,25,37,37,58,26,21,20
2020-09-14,34,48,99,35,47,53,63,26,17,12
2020-09-21,40,54,114,36,60,30,90,49,19,30
2020-09-28,48,63,111,38,67,30,80,38,19,27
2020-10-05,45,54,129,37,63,49,56,28,11,16
2020-10-12,32,32,80,36,57,37,61,30,31,11
2020-10-19,28,74,113,31,67,47,79,32,19,19
2020-10-26,42,62,98,40,68,62,54,29,28,19
2020-11-02,50,52,105,36,50,73,70,36,19,11
2020-11-09,30,43,58,31,47,38,40,29,18,21
2020-11-16,27,40,72,27,42,42,53,40,15,15
2020-11-23,20,25,54,19,36,34,40,25,15,17
2020-11-30,28,64,88,36,48,42,89,45,20,23
2020-12-07,39,38,92,15,32,35,38,26,9,12
2020-12-14,31,35,89,30,47,45,34,22,11,13
2020-12-21,21,17,51,34,28,19,32,15,7,14
2020-12-28,20,17,48,18,39,20,33,17,6,21
2021-01-04,52,54,112,40,56,86,51,30,21,27
2021-01-11,64,72,99,28,66,38,37,38,33,21
2021-01-18,51,70,84,26,54,33,55,37,16,16
2021-01-25,30,47,64,38,62,33,49,29,13,11
2021-02-01,72,93,127,21,80,45,62,32,22,26
2021-02-08,36,57,101,42,87,49,83,57,21,27
2021-02-15,66,65,89,31,90,46,42,25,10,28
2021-02-22,32,70,154,42,64,51,67,23,13,18
2021-03-01,49,64,130,35,79,65,100,32,25,21
2021-03-08,52,60,93,32,63,29,72,37,29,27
2021-03-15,57,39,124,25,71,32,86,36,23,32
2021-03-22,57,56,105,29,73,58,77,33,28,20
2021-03-29,49,75,116,34,60,60,89,34,18,41
2021-04-05,56,63,112,23,83,46,104,35,14,27
2021-04-12,39,67,72,29,77,45,67,44,19,25
2021-04-19,65,79,118,30,78,74,81,33,29,32
2021-04-26,68,71,99,45,69,61,50,39,31,26
2021-05-03,64,83,131,30,82,72,77,41,22,22
2021-05-10,37,70,120,32,138,42,75,37,20,31
2021-05-17,42,46,205,36,93,52,97,38,22,18
2021-05-24,43,66,226,49,86,50,92,30,25,23
2021-05-31,74,49,253,43,169,44,79,41,11,35
2021-06-07,70,76,295,29,178,56,97,39,24,35
2021-06-14,49,56,207,42,169,41,73,17,11,21
2021-06-21,54,107,172,18,185,56,87,36,22,42
2021-06-28,69,72,171,54,183,57,111,42,22,32
2021-07-05,59,56,151,38,143,38,128,34,19,22
2021-07-12,64,66,203,44,147,54,109,42,23,11
2021-07-19,72,103,167,32,161,43,128,41,24,25
2021-07-26,63,91,158,34,128,43,134,41,37,37
2021-08-02,48,84,141,28,96,45,148,50,22,28
2021-08-09,30,19,57,24,19,28,28,22,10,26
2021-08-16,39,43,98,35,40,20,28,20,14,16
2021-08-23,44,74,91,43,47,32,68,25,29,29
2021-08-30,47,49,110,56,53,57,76,17,16,18
2021-09-06,30,49,60,36,53,26,57,30,12,16
2021-09-13,45,53,80,21,42,36,51,28,13,17
2021-09-20,39,39,92,20,46,55,41,25,21,32
2021-09-27,66,42,91,49,46,44,61,38,30,27
2021-10-04,38,70,125,32,49,49,64,31,14,21
2021-10-11,35,34,68,38,23,38,41,25,16,26
2021-10-18,27,41,72,25,20,35,52,22,9,21
2021-10-25,40,45,71,38,39,33,83,43,22,19
2021-11-01,60,58,69,35,56,38,66,52,20,12
2021-11-08,65,37,104,30,38,16,60,18,24,28
2021-11-15,42,57,97,32,43,20,86,49,29,20
2021-11-22,32,31,80,17,30,29,54,28,18,14
2021-11-29,60,55,121,33,37,39,71,35,9,7
2021-12-06,45,58,115,35,61,32,62,46,16,15
2021-12-13,15,35,78,14,40,29,33,21,8,11
2021-12-20,15,21,51,17,19,22,16,17,12,9
2021-12-27,47,26,70,15,24,26,21,9,12,9
2022-01-03,72,47,83,27,39,43,29,23,15,23
2022-01-10,35,33,84,41,33,36,41,19,18,12
2022-01-17,98,36,92,24,49,33,54,29,20,16
2022-01-24,75,44,94,29,36,28,49,18,21,19
2022-01-31,57,46,111,25,58,55,56,30,16,14
2022-02-07,41,38,108,36,64,20,34,40,33,23
2022-02-14,46,35,85,30,30,35,64,21,19,19
2022-02-21,49,42,69,27,52,29,57,27,27,15
2022-02-28,48,27,91,25,51,34,70,40,15,20
2022-03-07,48,60,101,33,62,41,53,25,30,26
2022-03-14,40,62,74,33,76,32,68,33,8,31
2022-03-21,62,62,99,30,50,55,52,29,17,22
2022-03-28,67,80,100,40,51,39,46,28,25,16
2022-04-04,46,65,99,18,61,32,79,17,15,17
2022-04-11,70,30,89,24,72,53,46,23,25,26
2022-04-18,44,31,92,40,51,13,52,29,18,15
2022-04-25,38,60,99,31,44,37,49,38,30,13
2022-05-02,45,43,101,29,41,30,92,51,36,15
2022-05-09,20,86,109,30,51,37,70,32,37,18
2022-05-16,27,48,101,22,47,37,79,48,41,16
2022-05-23,67,34,99,29,34,34,43,21,31,13
2022-05-30,37,29,146,24,40,51,49,23,20,16
2022-06-06,33,72,105,25,38,29,62,37,17,13
2022-06-13,44,54,106,28,39,48,80,37,13,15
2022-06-20,37,38,85,20,43,27,51,17,11,14
2022-06-27,43,33,99,35,51,50,58,27,17,9
2022-07-04,48,45,118,31,34,49,45,40,41,31
2022-07-11,46,43,99,34,33,34,46,33,18,22
2022-07-18,48,66,81,17,52,54,65,48,5,37
2022-07-25,28,57,98,32,57,43,49,38,18,26
2022-08-01,58,61,135,27,53,58,45,31,31,24
2022-08-08,88,47,82,27,49,24,54,37,20,23
2022-08-15,72,52,79,24,40,61,94,37,26,28
2022-08-22,47,68,89,27,33,42,51,48,20,15
2022-08-29,61,72,81,44,42,27,49,25,16,17
2022-09-05,28,43,75,19,36,34,39,29,18,14
2022-09-12,30,66,95,23,51,56,44,31,19,19
2022-09-19,31,53,86,22,41,55,46,40,26,18
2022-09-26,48,37,111,12,41,49,65,24,36,24
2022-10-03,40,27,99,45,36,60,95,30,26,24
2022-10-10,61,53,85,46,51,36,48,37,19,23
2022-10-17,53,46,85,33,42,52,52,32,31,20
2022-10-24,29,27,114,30,40,40,64,36,25,22
2022-10-31,34,29,93,39,44,28,73,27,19,13
2022-11-07,37,55,69,25,33,56,46,26,22,27
2022-11-14,60,57,88,45,58,40,42,26,27,18
2022-11-21,45,25,63,24,31,25,34,19,17,8
2022-11-28,57,81,60,29,54,53,65,59,8,12
2022-12-05,35,29,132,31,60,39,73,40,22,22
2022-12-12,61,66,131,37,71,38,56,33,14,20
2022-12-19,32,27,72,27,25,36,48,24,14,21
2022-12-26,20,37,44,30,51,21,41,23,18,16
2023-01-02,41,36,66,44,39,49,49,19,18,26
2023-01-09,36,31,89,25,50,27,54,20,13,14
2023-01-16,52,87,99,37,47,38,63,35,36,11
2023-01-23,86,83,111,26,43,38,86,29,37,15
2023-01-30,75,67,127,28,55,46,61,22,21,29
2023-02-06,35,100,132,32,83,44,87,27,43,30
2023-02-13,67,81,143,24,64,49,72,47,18,17
2023-02-20,48,70,60,19,29,42,26,21,24,22
2023-02-27,57,72,80,24,51,27,49,25,32,25
2023-03-06,64,54,103,29,78,36,61,18,23,25
2023-03-13,43,57,118,30,48,50,59,30,16,11
2023-03-20,66,53,81,39,47,58,50,29,19,16
2023-03-27,36,56,115,36,68,48,46,34,9,19
2023-04-03,90,75,116,40,69,39,53,30,15,26
2023-04-10,33,50,97,53,52,36,104,29,15,7
2023-04-17,31,76,144,67,87,53,72,38,32,21
2023-04-24,61,72,96,46,106,85,110,41,29,37
2023-05-01,45,70,159,28,53,50,62,36,12,19
2023-05-08,57,55,91,29,65,51,65,51,45,31
2023-05-15,45,83,129,40,92,52,113,36,27,30
2023-05-22,44,69,131,39,51,48,70,49,26,32
2023-05-29,71,59,106,27,77,55,55,46,19,33
2023-06-05,60,72,136,40,65,49,44,37,28,20
2023-06-12,30,55,113,54,95,48,63,52,35,33
2023-06-19,60,36,84,40,117,47,50,42,24,18
2023-06-26,48,56,95,27,46,62,45,27,26,29
2023-07-03,56,49,112,38,48,48,48,43,17,14
2023-07-10,57,64,151,25,44,60,42,17,16,22
2023-07-17,58,66,117,49,64,50,51,26,19,32
2023-07-24,60,76,90,42,81,71,56,34,31,50
2023-07-31,72,66,103,33,61,75,50,41,31,33
2023-08-07,56,60,117,26,87,45,49,35,14,33
2023-08-14,52,48,90,35,69,62,62,41,25,26
2023-08-21,48,53,91,38,67,54,49,41,15,36
2023-08-28,55,69,139,41,41,50,53,57,20,29
2023-09-04,63,39,133,28,64,48,45,38,13,23
2023-09-11,66,49,94,41,94,60,40,33,20,37
2023-09-18,43,82,81,31,117,47,51,35,17,26
2023-09-25,94,90,84,62,96,45,49,32,25,40
2023-10-02,79,53,91,66,107,45,53,47,20,22
2023-10-09,50,67,83,45,98,67,82,54,13,17
2023-10-16,58,60,126,34,100,36,37,55,15,17
2023-10-23,82,91,79,46,88,68,39,72,21,20
2023-10-30,76,83,128,35,76,75,46,35,16,38
2023-11-06,27,92,102,46,96,53,37,32,20,24
2023-11-13,58,61,98,57,107,74,52,43,20,27
2023-11-20,41,48,56,27,36,43,33,25,17,25
2023-11-27,37,53,133,23,36,59,58,33,12,28
2023-12-04,66,46,123,39,87,42,67,18,36,30
2023-12-11,63,42,107,26,70,51,41,27,17,13
2023-12-18,40,46,96,26,63,30,23,26,20,20
2023-12-25,36,38,66,33,40,51,54,43,7,18
2024-01-01,59,69,131,29,76,39,57,45,9,25
2024-01-08,133,55,113,22,107,46,82,52,16,26
2024-01-15,83,94,118,28,107,49,74,33,12,52
2024-01-22,100,114,103,35,103,59,96,55,21,35
2024-01-29,77,46,104,38,80,44,89,38,31,16
2024-02-05,32,15,49,45,35,28,19,21,11,16
2024-02-12,53,40,64,33,53,27,23,12,14,24
2024-02-19,22,31,47,13,34,26,30,15,11,11
2024-02-26,55,59,118,41,57,51,101,47,17,26
2024-03-04,83,108,126,29,84,50,58,51,26,18
2024-03-11,83,77,130,39,69,34,79,53,25,28
2024-03-18,83,60,130,32,153,71,72,58,24,26
2024-03-25,50,70,124,44,54,53,59,53,18,12
2024-04-01,81,61,120,31,79,52,75,49,15,14
2024-04-08,79,83,176,38,66,57,83,30,13,31
2024-04-15,84,75,162,31,66,77,75,57,25,25
2024-04-22,54,55,132,39,87,58,90,24,22,25
2024-04-29,68,41,123,34,85,77,66,38,10,31
2024-05-06,62,27,152,52,71,60,50,41,23,24
2024-05-13,47,54,119,38,83,52,69,36,14,24
2024-05-20,64,69,125,35,94,47,83,47,13,29
2024-05-27,44,46,195,26,101,41,81,35,16,14
2024-06-03,67,57,167,28,122,35,59,61,22,25
2024-06-10,76,38,190,30,86,71,94,45,22,23
2024-06-17,45,39,138,24,49,54,70,25,18,18
2024-06-24,100,57,146,47,73,58,74,56,25,34
2024-07-01,32,59,115,33,41,49,60,33,11,23
2024-07-08,74,69,169,43,73,53,76,34,33,38
2024-07-15,78,52,116,41,60,70,59,50,18,24
2024-07-22,77,51,121,41,58,58,85,48,28,31
2024-07-29,57,76,227,50,86,66,78,54,28,39
2024-08-05,45,41,167,36,66,79,81,39,30,26
2024-08-12,67,38,117,39,77,88,98,38,29,28
2024-08-19,55,54,130,36,64,90,70,47,29,35
2024-08-26,70,59,134,54,56,81,70,39,30,22
2024-09-02,88,60,158,44,97,51,59,36,23,13
2024-09-09,64,38,163,45,67,63,62,32,32,27
2024-09-16,49,95,167,41,117,63,60,55,22,26
2024-09-23,78,102,159,45,84,49,68,48,18,28
2024-09-30,86,48,128,40,78,42,64,46,21,13
2024-10-07,44,58,133,26,44,52,72,42,15,18
2024-10-14,53,43,95,50,65,54,95,42,14,14
2024-10-21,66,56,148,47,81,54,93,38,27,27
2024-10-28,45,67,175,31,87,57,68,48,21,41
2024-11-04,56,58,143,40,80,49,92,31,16,33
2024-11-11,78,39,102,31,63,49,70,26,21,26
2024-11-18,72,35,127,26,48,43,57,42,17,24
2024-11-25,40,39,65,13,39,33,52,16,12,24
2024-12-02,50,54,166,33,85,81,42,31,21,36
2024-12-09,79,48,139,27,46,55,58,51,21,16
2024-12-16,65,31,113,30,56,47,33,25,20,36
2024-12-23,29,28,110,18,30,27,32,26,18,26
2024-12-30,29,41,116,21,57,41,38,20,11,20
2025-01-06,43,68,164,38,68,70,52,39,36,27
2025-01-13,61,55,97,26,56,54,50,48,34,23
2025-01-20,67,60,136,26,66,63,58,32,29,25
2025-01-27,62,46,98,26,58,41,64,41,39,25
2025-02-03,69,54,94,29,59,48,55,36,23,27
2025-02-10,48,67,103,33,53,52,68,33,17,12
2025-02-17,46,56,118,52,60,55,56,32,18,24
2025-02-24,62,88,123,37,58,58,77,47,26,30
2025-03-03,62,77,148,40,83,63,74,48,28,25
2025-03-10,44,48,141,47,62,48,68,41,17,24
2025-03-17,57,59,141,51,65,48,51,35,18,26
2025-03-24,68,44,120,34,69,44,64,26,31,24
2025-03-31,52,55,104,23,54,44,44,38,18,19
2025-04-07,53,43,117,50,56,56,58,43,21,14
2025-04-14,41,59,118,43,65,61,65,39,31,44
2025-04-21,88,52,128,34,39,56,82,58,21,18
2025-04-28,86,76,121,72,84,59,64,60,23,36
2025-05-05,69,78,168,45,97,50,59,38,26,35
2025-05-12,78,54,111,63,105,52,70,46,32,52
2025-05-19,63,49,111,57,60,65,67,44,36,46
2025-05-26,59,42,162,63,75,76,58,49,22,29
2025-06-02,60,44,162,37,108,65,61,33,26,44
2025-06-09,76,72,121,49,112,52,45,29,32,40
2025-06-16,44,38,88,33,117,54,78,22,27,54
2025-06-23,48,66,140,45,91,66,86,38,16,24
2025-06-30,44,50,166,64,50,66,85,43,19,22
2025-07-07,51,53,221,73,78,76,104,36,29,23
2025-07-14,46,53,208,72,63,81,79,43,29,37
2025-07-21,92,58,153,58,80,58,82,32,20,19
2025-07-28,93,56,164,65,91,79,56,56,23,18
2025-08-04,130,53,225,65,76,87,91,40,21,39
2025-08-11,129,57,143,50,119,70,91,51,37,35
2025-08-18,102,56,152,73,100,75,77,31,29,23
2025-08-25,131,61,150,82,87,82,75,41,33,40
2025-09-01,94,53,147,48,78,66,80,33,33,35
2025-09-08,79,31,140,46,72,65,80,28,24,26
2025-09-15,73,28,145,41,88,74,73,48,16,25
2025-09-22,56,61,154,57,76,57,103,33,25,40
2025-09-29,91,46,189,41,81,47,89,48,25,51
2025-10-06,115,41,142,61,92,70,69,39,22,33
2025-10-13,55,55,125,39,82,34,78,32,38,21
2025-10-20,111,40,164,46,49,58,70,46,37,24
2025-10-27,78,72,156,34,58,85,61,31,40,56
2025-11-03,85,58,192,52,91,74,66,65,39,52
2025-11-10,83,59,116,55,105,44,74,51,28,21
2025-11-17,70,41,170,56,56,60,49,43,15,20
2025-11-24,89,38,113,33,77,50,44,42,26,19
2025-12-01,75,60,162,34,101,84,63,34,32,46
2025-12-08,85,62,148,47,81,87,58,37,16,25
2025-12-15,49,70,111,44,71,74,56,39,32,22
2025-12-22,55,37,76,25,54,42,40,32,15,15
"""
weekly = pd.read_csv(StringIO(weekly_csv), parse_dates=["week_start"])
weekly.tail()

,week_start,1,2,3,4,5,6,7,8,9,10
412,2025-11-24,89,38,113,33,77,50,44,42,26,19
413,2025-12-01,75,60,162,34,101,84,63,34,32,46
414,2025-12-08,85,62,148,47,81,87,58,37,16,25
415,2025-12-15,49,70,111,44,71,74,56,39,32,22
416,2025-12-22,55,37,76,25,54,42,40,32,15,15


### Build features using completed weeks
A lag is a count from an earlier week in the same district. Shift before averaging so the target week cannot enter its own predictors.

In [3]:
panel = weekly.melt("week_start", var_name="district", value_name="count")
panel["district"] = panel.district.astype(int)
panel = panel.sort_values(["district", "week_start"]).reset_index(drop=True)
history = panel.groupby("district")["count"]
for lag in (1, 2, 4):
    panel[f"lag_{lag}"] = history.shift(lag)
for window in (4, 13):
    panel[f"rolling_{window}"] = history.transform(lambda x: x.shift(1).rolling(window).mean())
phase = 2 * np.pi * panel.week_start.dt.dayofyear / 365.25
panel["sin_year"], panel["cos_year"] = np.sin(phase), np.cos(phase)
panel["week_end"] = panel.week_start + pd.Timedelta(days=6)
panel = panel.dropna()
numeric = ["lag_1", "lag_2", "lag_4", "rolling_4", "rolling_13", "sin_year", "cos_year"]
inputs = ["district", *numeric]

### Fit the model and predict one week ahead
Use complete weeks starting in 2020. Fit preprocessing on the training rows. Each scored week uses the history that would have been available at its start.

In [4]:
def forecast(year):
    train = panel.loc[(panel.week_start >= "2020-01-01") & (panel.week_end.dt.year < year)]
    test = panel.loc[(panel.week_start.dt.year == year) & (panel.week_end.dt.year == year)].copy()
    preprocess = ColumnTransformer([
        ("district", OneHotEncoder(handle_unknown="error", sparse_output=False), ["district"]),
        ("numeric", StandardScaler(), numeric),
    ])
    model = make_pipeline(preprocess, PoissonRegressor(alpha=1.0, max_iter=2000, tol=1e-8))
    model.fit(train[inputs], train["count"])
    test["Poisson"] = model.predict(test[inputs])
    test["Previous week"] = test.lag_1
    test["Four-week average"] = test.rolling_4
    print(f"{year}: {len(train):,} training rows, {len(test)} scored district-weeks")
    return test

def scores(test):
    errors = test[["Poisson", "Previous week", "Four-week average"]].sub(test["count"], axis=0)
    return pd.DataFrame({"MAE": errors.abs().mean(), "RMSE": np.sqrt(errors.pow(2).mean())}).round(2)

validation = forecast(2024)
scores(validation)

2024: 2,080 training rows, 520 scored district-weeks


,MAE,RMSE
Poisson,13.64,19.13
Previous week,15.80,22.27
Four-week average,15.03,20.71


### Inspect a single forecast
Change the district or Monday below to explore another validation week.

In [5]:
district, monday = 3, "2024-07-29"
case = validation.loc[(validation.district == district) & (validation.week_start == monday)].iloc[0]
print(f"District {district}, week of {monday}: about {case['Poisson']:.0f} predicted, {case['count']:.0f} observed.")

District 3, week of 2024-07-29: about 123 predicted, 227 observed.


### Reproduce the recorded 2025 test
Refit the same settings through 2024, then score the 51 complete weeks in 2025. This replay does not tune on test outcomes or overwrite any saved project results.

In [6]:
test = forecast(2025)
final = scores(test)
assert len(test) == 510 and len(validation) == 520
assert np.allclose(final.loc["Poisson"], [12.59, 17.53], atol=0.01)
final

2025: 2,600 training rows, 510 scored district-weeks


,MAE,RMSE
Poisson,12.59,17.53
Previous week,13.97,19.07
Four-week average,12.95,17.25
